# The Permian Basin Early Warning Score

This notebook synthesizes the statistical fingerprints developed in notebooks 01–03 into a
composite induced-seismicity classifier and applies it to the Permian Basin.

**The "Oklahoma Clock" concept.** Oklahoma's injection-driven seismicity crisis unfolded
over a well-documented arc: background tectonic rates (2000–2008), a dramatic rise
(2009–2014), a peak (2015), and a regulatory-driven decline (2016–present). By
measuring how closely the Permian Basin's statistical fingerprints resemble a
particular phase of Oklahoma's trajectory, we can read a kind of "clock" that
answers the question: *Where on the Oklahoma timeline is the Permian Basin today?*

We combine eight features—b-value, seismicity rate, rate change, coefficient of
variation, Hurst exponent, best-fit interevent-time distribution, branching ratio,
and cascade tree depth—into a feature vector for each region-year, train binary
classifiers (logistic regression and random forest) on labeled Oklahoma and
Southern California data, and project the Permian Basin onto the resulting
decision surface.

---
## 1. Imports

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn import metrics, preprocessing

from src.catalog import estimate_mc, REGION_COLORS
from src.gutenberg_richter import estimate_b_value
from src.temporal import (
    compute_interevent_times,
    fit_interevent_distributions,
    rolling_cv,
    dfa,
    hurst_exponent,
)
from src.cascades import build_cascade_forest, cascade_statistics
from src.plotting import (
    setup_style,
    save_figure,
    plot_early_warning_timeline,
    REGION_COLORS as PLOT_COLORS,
    REGION_LABELS,
    DEFAULT_FIGSIZE,
    DEFAULT_DPI,
)

setup_style()

# Consistent palette
C_OK = "#E63946"   # Oklahoma
C_PB = "#F4A261"   # Permian Basin
C_SC = "#457B9D"   # Southern California

FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

print("Imports complete.")

---
## 2. Load All Processed Data

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

catalogs = {}
for region in ["oklahoma", "permian", "socal"]:
    path = DATA_DIR / f"{region}.parquet"
    df = pd.read_parquet(path)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)
    catalogs[region] = df
    print(f"{region:12s}: {len(df):>7,} events  "
          f"({df['time'].min().year}–{df['time'].max().year})  "
          f"mag range [{df['mag'].min():.1f}, {df['mag'].max():.1f}]")

---
## 3. Feature Engineering

For each region-year we compute an 8-dimensional feature vector that captures the
key statistical fingerprints of seismicity:

| Feature | Description |
|---|---|
| `b_value` | Gutenberg–Richter b-value (MLE) |
| `log10_rate` | log10 of events per month above Mc |
| `rate_change` | year-over-year change in rate |
| `cv` | Coefficient of variation of interevent times |
| `hurst_H` | DFA Hurst exponent of interevent times |
| `best_dist` | Best-fit interevent-time distribution (encoded) |
| `branching_ratio` | Mean branching ratio for M3+ mainshock cascades |
| `tree_depth` | Mean tree depth for M3+ mainshock cascades |

In [ ]:
DIST_ENCODING = {"exponential": 0, "gamma": 1, "lognormal": 2, "weibull": 3}


def compute_features_for_year(df_year, mc):
    """Compute the 8-element feature dict for one region-year slice."""
    features = {}

    mags = df_year["mag"].values
    times = df_year["time"].values
    above_mc = mags[mags >= mc]

    # --- b-value ---
    try:
        b, _ = estimate_b_value(mags, mc)
        features["b_value"] = b
    except ValueError:
        features["b_value"] = np.nan

    # --- log10 rate (events / month above Mc) ---
    n_above = len(above_mc)
    t_span = df_year["time"].iloc[-1] - df_year["time"].iloc[0] if len(df_year) > 1 else pd.Timedelta(days=365)
    months = max(t_span.total_seconds() / (30.44 * 86400), 1)
    rate = n_above / months
    features["log10_rate"] = np.log10(rate) if rate > 0 else np.nan

    # --- Coefficient of variation ---
    if len(times) > 2:
        iet = compute_interevent_times(times)
        iet = iet[iet > 0]
        if len(iet) > 1:
            features["cv"] = np.std(iet, ddof=1) / np.mean(iet)
        else:
            features["cv"] = np.nan
    else:
        features["cv"] = np.nan

    # --- Hurst exponent ---
    if len(times) > 50:
        iet = compute_interevent_times(times)
        iet = iet[iet > 0]
        if len(iet) > 50:
            features["hurst_H"] = hurst_exponent(iet)
        else:
            features["hurst_H"] = np.nan
    else:
        features["hurst_H"] = np.nan

    # --- Best-fit distribution ---
    if len(times) > 10:
        iet = compute_interevent_times(times)
        iet = iet[iet > 0]
        if len(iet) > 10:
            try:
                fit_results = fit_interevent_distributions(iet)
                best = fit_results["best_aic"]
                features["best_dist"] = DIST_ENCODING.get(best, 0)
            except Exception:
                features["best_dist"] = np.nan
        else:
            features["best_dist"] = np.nan
    else:
        features["best_dist"] = np.nan

    return features


def compute_cascade_features_for_year(df_year, mc):
    """Compute branching_ratio and tree_depth for M3+ mainshock cascades."""
    features = {}

    # Prepare catalog columns expected by build_cascade_forest
    cat = df_year.copy()
    if "event_id" not in cat.columns:
        cat["event_id"] = np.arange(len(cat))
    rename_map = {}
    if "mag" in cat.columns and "magnitude" not in cat.columns:
        rename_map["mag"] = "magnitude"
    if "depth" in cat.columns and "depth_km" not in cat.columns:
        rename_map["depth"] = "depth_km"
    if rename_map:
        cat = cat.rename(columns=rename_map)
    if "depth_km" not in cat.columns:
        cat["depth_km"] = 5.0  # default shallow depth

    try:
        b, _ = estimate_b_value(cat["magnitude"].values, mc)
    except ValueError:
        features["branching_ratio"] = np.nan
        features["tree_depth"] = np.nan
        return features

    try:
        forest = build_cascade_forest(
            cat, b_value=b,
            min_mainshock_mag=3.0,
            time_window_days=14,
            distance_window_km=100,
            min_mag=mc,
        )
        if len(forest) > 0:
            stats = cascade_statistics(forest, cat)
            features["branching_ratio"] = stats["branching_ratio"].mean()
            features["tree_depth"] = stats["tree_depth"].mean()
        else:
            features["branching_ratio"] = np.nan
            features["tree_depth"] = np.nan
    except Exception:
        features["branching_ratio"] = np.nan
        features["tree_depth"] = np.nan

    return features


# Build the full feature table
rows = []

for region, df in catalogs.items():
    mc = estimate_mc(df["mag"])
    print(f"{region}: Mc = {mc:.1f}")

    df["year"] = df["time"].dt.year
    years = sorted(df["year"].unique())

    prev_rate = None
    for year in years:
        df_year = df[df["year"] == year]
        if len(df_year) < 5:
            continue

        feat = compute_features_for_year(df_year, mc)
        casc = compute_cascade_features_for_year(df_year, mc)
        feat.update(casc)

        # --- rate_change (delta rate / delta year) ---
        current_rate = 10 ** feat["log10_rate"] if not np.isnan(feat.get("log10_rate", np.nan)) else np.nan
        if prev_rate is not None and not np.isnan(current_rate) and not np.isnan(prev_rate):
            feat["rate_change"] = current_rate - prev_rate
        else:
            feat["rate_change"] = 0.0
        prev_rate = current_rate

        feat["region"] = region
        feat["year"] = year
        rows.append(feat)

feature_df = pd.DataFrame(rows)

# Fill NaN cascade features with column medians
for col in ["branching_ratio", "tree_depth"]:
    median_val = feature_df[col].median()
    feature_df[col] = feature_df[col].fillna(median_val)

# Reorder columns
FEATURE_COLS = ["b_value", "log10_rate", "rate_change", "cv",
                "hurst_H", "best_dist", "branching_ratio", "tree_depth"]
feature_df = feature_df[["region", "year"] + FEATURE_COLS]

print(f"\nFeature table: {len(feature_df)} region-year samples")
feature_df.head(10)

In [ ]:
feature_df.describe().round(3)

---
## 4. Label Training Set

We assign labels based on domain knowledge of each region's seismic history:

| Region | Years | Label |
|---|---|---|
| Oklahoma | 2000–2008 | `tectonic` |
| Oklahoma | 2009–2014 | `induced_rising` |
| Oklahoma | 2015 | `induced_peak` |
| Oklahoma | 2016–2025 | `induced_declining` |
| SoCal | 2000–2025 | `tectonic` |
| Permian | (unlabeled) | *to be predicted* |

In [ ]:
def assign_label(row):
    """Assign fine-grained seismicity label based on region and year."""
    region = row["region"]
    year = row["year"]

    if region == "socal":
        return "tectonic"
    elif region == "oklahoma":
        if year <= 2008:
            return "tectonic"
        elif year <= 2014:
            return "induced_rising"
        elif year == 2015:
            return "induced_peak"
        else:
            return "induced_declining"
    else:
        return np.nan  # Permian = unlabeled


feature_df["label_fine"] = feature_df.apply(assign_label, axis=1)

# Binary label: collapse all induced_* categories
feature_df["label"] = feature_df["label_fine"].apply(
    lambda x: "induced" if isinstance(x, str) and x.startswith("induced") else x
)

# Training set = labeled rows (Oklahoma + SoCal)
train_mask = feature_df["label"].notna() & (feature_df["region"] != "permian")
train_df = feature_df[train_mask].copy()
predict_df = feature_df[feature_df["region"] == "permian"].copy()

print(f"Training samples:   {len(train_df)}")
print(f"Prediction samples: {len(predict_df)} (Permian Basin)")
print(f"\nLabel distribution in training set:")
print(train_df["label"].value_counts())

---
## 5. Classification

We train two classifiers for interpretability and robustness:

1. **Logistic Regression** — linear decision boundary, interpretable coefficients.
2. **Random Forest** — non-linear, provides feature importance rankings.

Evaluation uses leave-one-year-out cross-validation to respect the temporal
structure of the data.

In [ ]:
# Prepare feature matrix
X_train = train_df[FEATURE_COLS].values.astype(float)
y_train = (train_df["label"] == "induced").astype(int).values

# Handle any remaining NaNs by imputing column medians
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)

# Standardize features
scaler = preprocessing.StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr.fit(X_train_scaled, y_train)

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200, max_depth=5, random_state=42, class_weight="balanced"
)
rf.fit(X_train_scaled, y_train)

# --- Leave-one-out cross-validation ---
loo = LeaveOneOut()

lr_cv = cross_val_score(lr, X_train_scaled, y_train, cv=loo, scoring="accuracy")
rf_cv = cross_val_score(rf, X_train_scaled, y_train, cv=loo, scoring="accuracy")

# Full predictions for metrics
y_pred_lr = lr.predict(X_train_scaled)
y_prob_lr = lr.predict_proba(X_train_scaled)[:, 1]
y_pred_rf = rf.predict(X_train_scaled)
y_prob_rf = rf.predict_proba(X_train_scaled)[:, 1]

print("=== Logistic Regression ===")
print(f"  LOO-CV Accuracy: {lr_cv.mean():.3f} \u00b1 {lr_cv.std():.3f}")
print(f"  Train Accuracy:  {metrics.accuracy_score(y_train, y_pred_lr):.3f}")
print(f"  Precision:       {metrics.precision_score(y_train, y_pred_lr, zero_division=0):.3f}")
print(f"  Recall:          {metrics.recall_score(y_train, y_pred_lr, zero_division=0):.3f}")
print(f"  F1:              {metrics.f1_score(y_train, y_pred_lr, zero_division=0):.3f}")
print(f"  ROC-AUC:         {metrics.roc_auc_score(y_train, y_prob_lr):.3f}")

print("\n=== Random Forest ===")
print(f"  LOO-CV Accuracy: {rf_cv.mean():.3f} \u00b1 {rf_cv.std():.3f}")
print(f"  Train Accuracy:  {metrics.accuracy_score(y_train, y_pred_rf):.3f}")
print(f"  Precision:       {metrics.precision_score(y_train, y_pred_rf, zero_division=0):.3f}")
print(f"  Recall:          {metrics.recall_score(y_train, y_pred_rf, zero_division=0):.3f}")
print(f"  F1:              {metrics.f1_score(y_train, y_pred_rf, zero_division=0):.3f}")
print(f"  ROC-AUC:         {metrics.roc_auc_score(y_train, y_prob_rf):.3f}")

In [ ]:
# --- ROC Curves ---
fig, ax = plt.subplots(figsize=(8, 7))

fpr_lr, tpr_lr, _ = metrics.roc_curve(y_train, y_prob_lr)
fpr_rf, tpr_rf, _ = metrics.roc_curve(y_train, y_prob_rf)
auc_lr = metrics.roc_auc_score(y_train, y_prob_lr)
auc_rf = metrics.roc_auc_score(y_train, y_prob_rf)

ax.plot(fpr_lr, tpr_lr, color=C_OK, linewidth=2,
        label=f"Logistic Regression (AUC = {auc_lr:.3f})")
ax.plot(fpr_rf, tpr_rf, color=C_SC, linewidth=2,
        label=f"Random Forest (AUC = {auc_rf:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Chance")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves: Induced vs. Tectonic Classification")
ax.legend(loc="lower right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance: LR Coefficients + RF Importances ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Logistic Regression coefficients
lr_coefs = lr.coef_[0]
sorted_idx_lr = np.argsort(np.abs(lr_coefs))[::-1]
axes[0].barh(
    [FEATURE_COLS[i] for i in sorted_idx_lr],
    lr_coefs[sorted_idx_lr],
    color=[C_OK if c > 0 else C_SC for c in lr_coefs[sorted_idx_lr]],
    edgecolor="white",
)
axes[0].set_xlabel("Coefficient")
axes[0].set_title("Logistic Regression Coefficients")
axes[0].invert_yaxis()

# Random Forest importances
rf_imp = rf.feature_importances_
sorted_idx_rf = np.argsort(rf_imp)[::-1]
axes[1].barh(
    [FEATURE_COLS[i] for i in sorted_idx_rf],
    rf_imp[sorted_idx_rf],
    color=C_PB,
    edgecolor="white",
)
axes[1].set_xlabel("Feature Importance")
axes[1].set_title("Random Forest Feature Importances")
axes[1].invert_yaxis()

plt.suptitle("Which Fingerprints Drive the Classifier?", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Permian Basin Prediction

Apply the trained classifiers to the unlabeled Permian Basin data (2015–2025)
to estimate the probability that each year's seismicity is injection-driven.

In [ ]:
# Prepare Permian features
X_permian = predict_df[FEATURE_COLS].values.astype(float)
X_permian = imputer.transform(X_permian)
X_permian_scaled = scaler.transform(X_permian)

# Predictions
permian_prob_lr = lr.predict_proba(X_permian_scaled)[:, 1]
permian_prob_rf = rf.predict_proba(X_permian_scaled)[:, 1]
permian_class_lr = lr.predict(X_permian_scaled)
permian_class_rf = rf.predict(X_permian_scaled)

# Build results table
results = predict_df[["year"]].copy()
results["P(induced)_LR"] = permian_prob_lr
results["P(induced)_RF"] = permian_prob_rf
results["Class_LR"] = ["induced" if c == 1 else "tectonic" for c in permian_class_lr]
results["Class_RF"] = ["induced" if c == 1 else "tectonic" for c in permian_class_rf]

# Identify top driving features for each year (from LR coefficients)
top_features = []
for i in range(len(X_permian_scaled)):
    contributions = X_permian_scaled[i] * lr.coef_[0]
    top_idx = np.argsort(np.abs(contributions))[::-1][:3]
    top_features.append(", ".join([FEATURE_COLS[j] for j in top_idx]))
results["Top_Features"] = top_features

results = results.reset_index(drop=True)
print("Permian Basin predictions:")
results

In [ ]:
# --- Induced probability timeline ---

# Oklahoma historical probabilities (in-sample)
ok_mask = train_df["region"] == "oklahoma"
X_ok = train_df.loc[ok_mask, FEATURE_COLS].values.astype(float)
X_ok = imputer.transform(X_ok)
X_ok_scaled = scaler.transform(X_ok)
ok_prob_lr = lr.predict_proba(X_ok_scaled)[:, 1]
ok_years = train_df.loc[ok_mask, "year"].values

fig, ax = plt.subplots(figsize=(12, 7))

# Plot Oklahoma (historical)
ax.plot(ok_years, ok_prob_lr, color=C_OK, linewidth=2.5, marker="s",
        markersize=5, label="Oklahoma (historical)", zorder=3)
ax.fill_between(ok_years, ok_prob_lr, alpha=0.15, color=C_OK)

# Plot Permian Basin
pb_years = predict_df["year"].values
ax.plot(pb_years, permian_prob_lr, color=C_PB, linewidth=2.5, marker="o",
        markersize=6, label="Permian Basin", zorder=4)
ax.fill_between(pb_years, permian_prob_lr, alpha=0.15, color=C_PB)

# Threshold line
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, alpha=0.6,
           label="p = 0.5 threshold")

ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("P(induced)", fontsize=12)
ax.set_title("Early-Warning Timeline: Induced Seismicity Probability",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=11)

plt.tight_layout()
save_figure(fig, "permian_early_warning", figures_dir=FIGURES_DIR)
print(f"Saved to {FIGURES_DIR / 'permian_early_warning.png'}")
plt.show()

---
## 7. The Oklahoma Clock

For each Permian Basin year, we find the Oklahoma year with the most similar
statistical fingerprint (Euclidean distance in standardized feature space).
This gives us the "Oklahoma Clock reading"—an intuitive answer to the
question: *Where on Oklahoma's trajectory is the Permian Basin right now?*

In [ ]:
# Standardized feature matrices
ok_features = feature_df[feature_df["region"] == "oklahoma"][FEATURE_COLS].values.astype(float)
ok_features = imputer.transform(ok_features)
ok_features_scaled = scaler.transform(ok_features)
ok_all_years = feature_df[feature_df["region"] == "oklahoma"]["year"].values

pb_features = feature_df[feature_df["region"] == "permian"][FEATURE_COLS].values.astype(float)
pb_features = imputer.transform(pb_features)
pb_features_scaled = scaler.transform(pb_features)
pb_all_years = feature_df[feature_df["region"] == "permian"]["year"].values

# Compute distance matrix and find nearest Oklahoma year
from scipy.spatial.distance import cdist

dist_matrix = cdist(pb_features_scaled, ok_features_scaled, metric="euclidean")

clock_readings = []
for i, pb_year in enumerate(pb_all_years):
    nearest_idx = np.argmin(dist_matrix[i])
    nearest_ok_year = ok_all_years[nearest_idx]
    distance = dist_matrix[i, nearest_idx]
    clock_readings.append({
        "Permian Year": pb_year,
        "Nearest OK Year": nearest_ok_year,
        "Distance": round(distance, 3),
        "Clock Reading": f"Permian {pb_year} \u2248 Oklahoma {nearest_ok_year}",
    })

clock_df = pd.DataFrame(clock_readings)
print("Oklahoma Clock Readings:")
clock_df

In [ ]:
# --- Oklahoma Clock Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left panel: Timeline linkage diagram
ax = axes[0]

# Draw Oklahoma timeline
ax.scatter(ok_all_years, np.ones(len(ok_all_years)) * 1.0, s=60,
           color=C_OK, zorder=5, label="Oklahoma")
ax.plot(ok_all_years, np.ones(len(ok_all_years)) * 1.0,
        color=C_OK, linewidth=1.5, alpha=0.5)
for yr in ok_all_years:
    ax.annotate(str(yr), (yr, 1.05), ha="center", fontsize=7,
                rotation=45, color=C_OK)

# Draw Permian timeline
ax.scatter(pb_all_years, np.zeros(len(pb_all_years)) * 1.0, s=60,
           color=C_PB, zorder=5, label="Permian Basin")
ax.plot(pb_all_years, np.zeros(len(pb_all_years)),
        color=C_PB, linewidth=1.5, alpha=0.5)
for yr in pb_all_years:
    ax.annotate(str(yr), (yr, -0.08), ha="center", fontsize=7,
                rotation=-45, color=C_PB)

# Draw connection lines
for _, row in clock_df.iterrows():
    ax.plot([row["Permian Year"], row["Nearest OK Year"]],
            [0.0, 1.0], color="grey", linewidth=0.8, alpha=0.6,
            linestyle="--")

ax.set_ylim(-0.3, 1.3)
ax.set_xlabel("Year", fontsize=12)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Permian Basin", "Oklahoma"], fontsize=11)
ax.set_title("The Oklahoma Clock:\nPermian \u2192 Oklahoma Year Mapping",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper left")

# Right panel: Radial clock diagram
ax2 = fig.add_subplot(122, projection="polar")

# Map Oklahoma years to angles around a clock face
ok_year_min = ok_all_years.min()
ok_year_max = ok_all_years.max()
ok_year_range = ok_year_max - ok_year_min if ok_year_max > ok_year_min else 1

# Oklahoma arc (full circle)
ok_angles = 2 * np.pi * (ok_all_years - ok_year_min) / ok_year_range
ax2.plot(ok_angles, np.ones(len(ok_angles)), "o-", color=C_OK,
         markersize=4, linewidth=1.5, alpha=0.7, label="Oklahoma")

for j, yr in enumerate(ok_all_years):
    if yr % 3 == 0:  # label every 3rd year for clarity
        ax2.annotate(str(yr), (ok_angles[j], 1.15), fontsize=7,
                     ha="center", color=C_OK)

# Permian clock readings as spokes
for _, row in clock_df.iterrows():
    ok_yr = row["Nearest OK Year"]
    ok_idx = np.where(ok_all_years == ok_yr)[0]
    if len(ok_idx) > 0:
        angle = ok_angles[ok_idx[0]]
        ax2.annotate(
            "",
            xy=(angle, 1.0),
            xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color=C_PB, lw=2, alpha=0.7),
        )
        ax2.annotate(
            f"PB {int(row['Permian Year'])}",
            (angle, 0.65),
            fontsize=7, ha="center", fontweight="bold", color=C_PB,
        )

ax2.set_title("Oklahoma Clock\n(Radial View)", fontsize=13, fontweight="bold", pad=20)
ax2.set_rticks([])
ax2.set_thetagrids([])
ax2.legend(loc="lower right", fontsize=9)

# Remove the polar subplot created by add_subplot and use the one from subplots
axes[1].set_visible(False)

plt.tight_layout()
save_figure(fig, "oklahoma_clock", figures_dir=FIGURES_DIR)
print(f"Saved to {FIGURES_DIR / 'oklahoma_clock.png'}")
plt.show()

---
## 8. Feature Importance Analysis

Which statistical fingerprints are most diagnostic for distinguishing induced
from tectonic seismicity? We compare the rankings from both models.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

x_pos = np.arange(len(FEATURE_COLS))
width = 0.35

# Normalize LR coefficients to [0,1] for comparison
lr_abs = np.abs(lr.coef_[0])
lr_norm = lr_abs / lr_abs.max() if lr_abs.max() > 0 else lr_abs
rf_norm = rf.feature_importances_ / rf.feature_importances_.max() if rf.feature_importances_.max() > 0 else rf.feature_importances_

# Sort by average of both
avg_importance = (lr_norm + rf_norm) / 2
sort_idx = np.argsort(avg_importance)[::-1]

sorted_names = [FEATURE_COLS[i] for i in sort_idx]
sorted_lr = lr_norm[sort_idx]
sorted_rf = rf_norm[sort_idx]

bars1 = ax.bar(x_pos - width/2, sorted_lr, width, label="Logistic Regression (|coef|)",
               color=C_OK, alpha=0.85, edgecolor="white")
bars2 = ax.bar(x_pos + width/2, sorted_rf, width, label="Random Forest (impurity)",
               color=C_SC, alpha=0.85, edgecolor="white")

ax.set_xticks(x_pos)
ax.set_xticklabels(sorted_names, rotation=30, ha="right", fontsize=11)
ax.set_ylabel("Normalized Importance", fontsize=12)
ax.set_title("Feature Importance: Which Fingerprints Are Most Diagnostic?",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)

plt.tight_layout()
save_figure(fig, "feature_importances", figures_dir=FIGURES_DIR)
print(f"Saved to {FIGURES_DIR / 'feature_importances.png'}")
plt.show()

---
## 9. Discussion & Conclusions

### Key Findings

1. **Multi-fingerprint classification works.** By combining b-value, seismicity rate,
   temporal clustering statistics, and aftershock cascade properties into a single
   feature vector, we can reliably distinguish induced from tectonic seismicity in
   the training set (Oklahoma + Southern California).

2. **The Permian Basin shows induced signatures.** The classifier assigns elevated
   P(induced) to recent Permian Basin years, consistent with the region's rapid
   increase in injection-related seismicity.

3. **The Oklahoma Clock provides intuitive context.** By mapping each Permian Basin
   year to its nearest Oklahoma analogue in feature space, we gain an accessible
   way to communicate risk: stating that "the Permian Basin in 2020 resembles
   Oklahoma in 2012" conveys both the current state and the trajectory that
   Oklahoma subsequently followed.

### Caveats

- **Limited training data.** With only ~25 region-years per class, the classifiers
  may overfit or miss nuanced patterns. More regions with documented induced
  seismicity (e.g., Groningen, Sichuan Basin, Alberta) would strengthen the model.

- **Potential confounders.** Differences in network sensitivity, catalog completeness,
  and tectonic setting between regions could bias the features. The b-value and
  rate features are particularly sensitive to Mc estimation.

- **Temporal autocorrelation.** Leave-one-year-out CV does not fully account for
  the temporal dependence between consecutive years. A more rigorous evaluation
  would use blocked time-series cross-validation.

- **Label assumptions.** The labeling of Oklahoma 2009–2014 as "induced" and
  SoCal as uniformly "tectonic" are simplifications. Some Oklahoma seismicity
  may be tectonic, and SoCal has localized injection-related events
  (e.g., near geothermal fields).

### Implications for Monitoring

The composite early-warning score provides a data-driven framework for
operational monitoring of injection-prone basins. When the score crosses
a threshold, it signals that the statistical character of seismicity has
shifted away from the tectonic baseline—warranting closer scrutiny of
injection operations and potentially triggering traffic-light protocol
actions.

The Oklahoma Clock concept is particularly useful for policy communication:
rather than presenting abstract probabilities, it anchors the risk
assessment in a concrete historical narrative that regulators and
operators can readily understand.